In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run utility path

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "order_items", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'dataset path{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")

In [0]:
display(df_bronze.filter(
    (F.col("price")<0.00) |
    (F.col("freight_value")<0.00)
    )
)

In [0]:
df_duplicates = df_bronze.groupBy("order_id","order_item_id") \
.count() \
.filter("count > 1")

df_duplicates.display()



In [0]:

df_bronze.select([
F.sum(F.col(c).isNull().cast("int")).alias(c)
for c in df_bronze.columns
]).display()

In [0]:
df_silver= df_bronze

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver= spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold= df_silver.select('order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value' )
df_gold.display()


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")